# Intervalo de confianza de Wilson para SIRAP

Este notebook calcula el IC score de Wilson al 95 % para proporciones binarias de un holdout ciego de SAVH con 57 imágenes puntuables. Presenta por separado la coincidencia exacta y la cobertura del OCR base y afinado.

Para x aciertos en n ensayos, la proporción observada es p̂ = x/n. El intervalo de Wilson se calcula con z = 1,96 al 95 %: centro = (p̂ + z²/(2n))/(1 + z²/n) y semiancho = z·√(p̂(1−p̂)/n + z²/(4n²))/(1 + z²/n). Los límites son centro ± semiancho, restringidos a [0, 1].

Cada ensayo debe ser una unidad independiente. Aquí la unidad es una imagen del holdout; el IC describe la incertidumbre de esas proporciones bajo el modelo binomial. No corrige errores de etiqueta, selección, cobertura ni dependencia entre observaciones, y no transforma este conjunto en una muestra representativa de otros escenarios. No se aplica a CER, latencia ni mAP, ni a cuadros repetidos de una misma pasada de video.


In [ ]:
from statistics import NormalDist

def wilson_interval(successes: int, trials: int, confidence_level: float = 0.95):
    """Devuelve el intervalo score de Wilson para una proporción binomial."""
    if isinstance(successes, bool) or isinstance(trials, bool):
        raise TypeError("successes y trials deben ser enteros")
    if not isinstance(successes, int) or not isinstance(trials, int):
        raise TypeError("successes y trials deben ser enteros")
    if trials <= 0 or not 0 <= successes <= trials:
        raise ValueError("Se requiere trials > 0 y 0 <= successes <= trials")
    if not 0 < confidence_level < 1:
        raise ValueError("confidence_level debe estar entre 0 y 1")
    p = successes / trials
    z = NormalDist().inv_cdf(0.5 + confidence_level / 2)
    z2 = z * z
    denom = 1 + z2 / trials
    center = (p + z2 / (2 * trials)) / denom
    half = z * (p * (1 - p) / trials + z2 / (4 * trials**2))**0.5 / denom
    return {"successes": successes, "trials": trials, "estimate": p,
            "confidence_level": confidence_level,
            "lower": max(0.0, center - half), "upper": min(1.0, center + half),
            "method": "wilson_score"}


In [ ]:
counts = {
    "OCR base: coincidencia exacta": (42, 57),
    "OCR afinado: coincidencia exacta": (35, 57),
    "OCR base: cobertura": (56, 57),
    "OCR afinado: cobertura": (56, 57),
}
for label, (successes, trials) in counts.items():
    ci = wilson_interval(successes, trials)
    print(f"{label}: {successes}/{trials} = {ci['estimate']:.1%}; "
          f"IC Wilson 95 % [{ci['lower']:.1%}, {ci['upper']:.1%}]")


## Lectura de los resultados

El OCR base obtuvo 42 aciertos exactos en 57 imágenes (73,7 %; IC de Wilson 95 %: 61,0–83,4 %). El OCR afinado obtuvo 35/57 (61,4 %; IC: 48,4–72,9 %). Ambos reconocieron o generaron una salida OCR en 56/57 imágenes (98,2 %; IC: 90,7–99,7 %). La cobertura significa que hubo una salida; no que la placa fuera correcta.

Los intervalos cuantifican la incertidumbre de cada proporción del holdout; por sí solos no prueban una diferencia estadísticamente significativa entre OCR base y afinado. Las 57 imágenes y sus conteos se mantienen como el conjunto evaluado, sin ajustes de representatividad.

El intervalo no se aplica directamente a métricas continuas como CER, latencia o mAP. Tampoco se calcula sobre lecturas repetidas de video como si cada cuadro fuera un ensayo independiente. Los conteos y resultados reproducibles están en [ocr_savh_wilson.json](../results/metrics/ocr_savh_wilson.json). La función general también está disponible en [src/evaluate.py](../src/evaluate.py).

Fuente metodológica: NIST/SEMATECH e-Handbook, [Wilson confidence intervals](https://www.itl.nist.gov/div898/handbook/prc/section2/prc241.htm).
